In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt




In [8]:
df = pd.read_csv('clean_data.csv', parse_dates=['Date'])
drinks = df.loc[df['Is_Drink'].eq(True)].copy()
plt.rcParams.update({'font.size': 10, 'axes.spines.top':False, 'axes.spines.right':False})



# Figure 1: Popular beverages
items = drinks['Item'].value_counts().head(10).sort_values()
fig,ax=plt.subplots(figsize=(9,5.4));ax.barh(items.index,items.values,color='#587da1');ax.set(xlabel='Sales records (not units sold)',title='Most frequently recorded beverages');fig.tight_layout();fig.savefig('01_top_drinks.png',dpi=180);plt.close(fig)



In [6]:
# Figure 2: normalized milk categories among explicitly recorded milk types, five highest volume drinks
chosen=drinks['Item'].value_counts().head(5).index
milk=drinks[drinks['Item'].isin(chosen)&drinks['Milk'].notna()]
cross=pd.crosstab(milk['Item'],milk['Milk']);cross=cross.div(cross.sum(axis=1),axis=0)*100
cross=cross.reindex(chosen).fillna(0)
fig,ax=plt.subplots(figsize=(10,5.5));cross.plot(kind='barh',stacked=True,ax=ax,colormap='tab20c',width=.75);ax.set(xlabel='% among records with a specified milk type',ylabel='',title='Recorded milk preferences vary across drink types',xlim=(0,100));ax.legend(title='Milk',loc='upper left',bbox_to_anchor=(1.01,1));fig.tight_layout();fig.savefig('02_milk_by_drink.png',dpi=180);plt.close(fig)




In [7]:
# Figure 3: split multi-syrup entries (each selected flavor can be counted once per record)
syrups=drinks['Syrup'].dropna().str.split(',').explode().str.strip().value_counts().head(10).sort_values()
fig,ax=plt.subplots(figsize=(9,5.3));ax.barh(syrups.index,syrups.values,color='#8571a7');ax.set(xlabel='Records mentioning this syrup (not exclusive)',title='Top syrup flavors (multi-flavor entries separated)');fig.tight_layout();fig.savefig('03_syrup_frequency.png',dpi=180);plt.close(fig)



In [9]:
# Figure 4: hourly total and temperature known fraction
hour_counts=drinks.groupby('Hour').size().reindex(range(9,16),fill_value=0)
known=drinks[drinks['Temp'].isin(['Hot','Iced'])]
ht=pd.crosstab(known['Hour'],known['Temp']).reindex(range(9,16),fill_value=0)
fig,ax=plt.subplots(figsize=(9,5));ax.plot(hour_counts.index,hour_counts.values,marker='o',linewidth=2,color='#387b83');ax.set(xticks=range(9,16),xlabel='Hour (24-hour clock)',ylabel='Number of beverage records',title='Beverage records by hour (9 AM–3 PM)');ax.grid(axis='y',alpha=.2);fig.tight_layout();fig.savefig('04_hourly_volume.png',dpi=180);plt.close(fig)



In [10]:
# Figure 5: temperature recording and breakdown by drink type (missing included)
chosen_temp=drinks['Item'].value_counts().head(6).index
d=drinks[drinks['Item'].isin(chosen_temp)].copy();d['Temperature recorded']=d['Temp'].fillna('Not recorded')
p=pd.crosstab(d['Item'],d['Temperature recorded'],normalize='index').reindex(chosen_temp)*100
p=p.reindex(columns=['Iced','Hot','Not recorded'],fill_value=0)
fig,ax=plt.subplots(figsize=(10,5.2));p.plot(kind='barh',stacked=True,ax=ax,color=['#67a8c7','#d98562','#b9bcc4']);ax.set(xlim=(0,100),xlabel='% of records',ylabel='',title='Temperature mix and missing labels by drink');ax.legend(title='Temperature',loc='upper left',bbox_to_anchor=(1.01,1));fig.tight_layout();fig.savefig('05_temperature_completeness.png',dpi=180);plt.close(fig)


In [ ]:

print('rows',len(df),'drinks',len(drinks),'milk_known',drinks.Milk.notna().sum(),'temp_known',drinks.Temp.notna().sum(),'syrup_known',drinks.Syrup.notna().sum())
print('top syrups',syrups.sort_values(ascending=False).to_dict())
print('hour totals',hour_counts.to_dict())
print('temperature matrix',p.round(1).to_string())